In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
import sys
sys.path.append(os.path.abspath(".."))

from baseline.generate import generate
import baseline.generate as gen_module
import torch

Loading tokenizer...
Loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Error while downloading from https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct/resolve/c03e6d358207e414f1eca0bb1891e29f1db0e242/model-00003-of-00004.safetensors: The read operation timed out
Trying to resume download...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(gen_module.model.parameters()).device)

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

CUDA available: True
Model device: cuda:0
memory.used [MiB], memory.total [MiB]
5565 MiB, 8188 MiB



In [5]:
from datasets import load_dataset
dataset = load_dataset("google-research-datasets/mbpp")

In [6]:
import time

sample = dataset["train"][0]

start = time.time()
response = generate(sample["text"], test_example=sample["test_list"][0])
elapsed = time.time() - start

print(f"Took {elapsed:.1f}s")
print(response)

Took 6.8s
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Solve the following problem in Python.

Problem:
Write a function to find the longest chain which can be formed from the given set of pairs.

Your code must satisfy this test case exactly (same function name, same argument order):
assert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3

If the test uses any class or object (e.g. Pair) that isn't a Python builtin, you must define that class yourself as part of your code.


IMPORTANT: Respond with ONLY a single Python code block. Do not include any explanation, reasoning, or text before or after the code block.
assistant
```python
class Pair:
    def __init__(self, start, end):
        self.start = start
        self.end = end

def max_chain_length(pairs, n):
    pairs.sort(key=lambda x: x.end)
    max_len = 1
    last_end = pairs[0].end
    for i in range(1, n):
        if pairs[i].start > last_end:
            max_

In [7]:
from evaluation.evaluator import evaluate, extract_code

result = evaluate(response, sample["test_list"], setup_code=sample.get("test_setup_code", ""))
print(result)
print(extract_code(response))

(True, None)
class Pair:
    def __init__(self, start, end):
        self.start = start
        self.end = end

def max_chain_length(pairs, n):
    pairs.sort(key=lambda x: x.end)
    max_len = 1
    last_end = pairs[0].end
    for i in range(1, n):
        if pairs[i].start > last_end:
            max_len += 1
            last_end = pairs[i].end
    return max_len


In [8]:
import json
from tqdm import tqdm

N = 30
results_7b_baseline = []
passed = 0

for i in tqdm(range(N)):
    sample = dataset["train"][i]
    try:
        response = generate(sample["text"], test_example=sample["test_list"][0])
    except Exception as e:
        results_7b_baseline.append({"idx": i, "passed": False, "error": f"generation error: {e}"})
        continue

    success, error = evaluate(
        response,
        sample["test_list"],
        setup_code=sample.get("test_setup_code", "")
    )

    if success:
        passed += 1

    results_7b_baseline.append({
        "idx": i,
        "task_id": sample.get("task_id", i),
        "passed": success,
        "error": error,
        "code": extract_code(response)
    })

    # save incrementally
    with open("../results/baseline_7b_run_30.json", "w") as f:
        json.dump(results_7b_baseline, f, indent=2)

print(f"\nPassed: {passed}/{N} ({passed/N*100:.1f}%)")
































100%|██████████| 30/30 [01:29<00:00,  2.97s/it]


Passed: 20/30 (66.7%)


In [ ]:
from baseline.generate import generate
from evaluation.evaluator import (
    extract_code, evaluate, get_defined_functions, get_called_functions, 
    get_function_name, run_candidate, deduplicate_candidates,
    extract_call_args, get_probe_inputs, find_disagreements, filter_invalid
)

def generate_candidates(problem_text, test_example, n=8, temperature=0.8):
    """Generate N diverse candidate solutions for one problem."""
    candidates = []
    for i in range(n):
        response = generate(
            problem_text,
            test_example=test_example,
            temperature=temperature,
            do_sample=True
        )
        code = extract_code(response)
        candidates.append(code)
    return candidates


def analyze_ambiguity(sample, n_candidates=8, temperature=0.8):
    """
    Full Phase 2 pipeline for a single MBPP sample.
    """
    problem_text = sample["text"]
    test_list = sample["test_list"]
    expected_func_name = get_called_functions(test_list)
    expected_func_name = list(expected_func_name)[0] if expected_func_name else None

    if expected_func_name is None:
        return {"error": "could not determine expected function name"}

    raw_candidates = generate_candidates(
        problem_text,
        test_example=test_list[0],
        n=n_candidates,
        temperature=temperature
    )

    valid_candidates = filter_invalid(raw_candidates)

    probe_inputs = get_probe_inputs(test_list)

    if not probe_inputs:
        return {
            "task_id": sample.get("task_id"),
            "note": "no simple-literal probe inputs available (custom class args?)",
            "n_valid_candidates": len(valid_candidates)
        }

    unique_candidates = deduplicate_candidates(
        valid_candidates, probe_inputs, expected_func_name
    )

    disagreements = find_disagreements(
        unique_candidates, probe_inputs, expected_func_name
    )

    return {
        "task_id": sample.get("task_id"),
        "n_raw_candidates": len(raw_candidates),
        "n_valid_candidates": len(valid_candidates),
        "n_unique_candidates": len(unique_candidates),
        "n_disagreement_points": len(disagreements),
        "disagreements": disagreements,
        "candidates": unique_candidates
    }

In [ ]:
import json

N = 15  
results_phase2_7b = []

for i in range(N):
    sample = dataset["train"][i]
    print(f"Analyzing task {sample.get('task_id', i)}...")
    result = analyze_ambiguity(sample, n_candidates=8, temperature=0.8)
    results_phase2_7b.append(result)

    with open("../results/phase2_ambiguity_7b.json", "w") as f:
        json.dump(results_phase2_7b, f, indent=2)

print("Done.")

Analyzing task 601...
Analyzing task 602...
Analyzing task 603...
Analyzing task 604...
Analyzing task 605...
Analyzing task 606...
Analyzing task 607...
Analyzing task 608...
Analyzing task 609...
Analyzing task 610...
Analyzing task 611...
Analyzing task 612...
Analyzing task 613...
Analyzing task 614...
Analyzing task 615...
Done.


In [16]:
with open("../results/phase2_ambiguity_7b.json", "r") as f:
    results_phase2_7b = json.load(f)

for i in range(15, 30):
    sample = dataset["train"][i]
    print(f"Analyzing task {sample.get('task_id', i)}...")
    result = analyze_ambiguity(sample, n_candidates=8, temperature=0.8)
    results_phase2_7b.append(result)

    with open("../results/phase2_ambiguity_7b.json", "w") as f:
        json.dump(results_phase2_7b, f, indent=2)

print("Done.")

Analyzing task 616...
Analyzing task 617...
Analyzing task 618...
Analyzing task 619...
Analyzing task 620...
Analyzing task 621...
Analyzing task 622...
Analyzing task 623...
Analyzing task 624...
Analyzing task 625...
Analyzing task 626...
Analyzing task 627...
Analyzing task 628...
Analyzing task 629...
Analyzing task 630...
Done.


In [17]:
import sys, os
sys.path.append(os.path.abspath(".."))

from evaluation.eig import (
    init_weights, compute_pt, eig_score, full_score, build_test_pool,
    oracle_answer, expected_pnext, should_ask, update_weights
)

def clarification_loop(sample, candidates, k_max=4, epsilon=0.02, gamma=0.85):
    func_name = list(get_called_functions(sample["test_list"]))[0]
    reference_code = sample["code"]

    weights = init_weights(candidates)

    probe_inputs = get_probe_inputs(sample["test_list"])
    disagreements = find_disagreements(candidates, probe_inputs, func_name)
    test_pool = build_test_pool(disagreements)

    questions_asked = []

    for _ in range(k_max):
        if not test_pool:
            break

        scored = []
        for (inp, hyp_output) in test_pool:
            pt, outcomes, defined_w = compute_pt(candidates, weights, run_candidate, func_name, inp, hyp_output)
            undefined_w = 1 - defined_w
            s = full_score(pt, defined_w, undefined_w, epsilon)
            scored.append((s, inp, hyp_output, outcomes))

        scored.sort(key=lambda x: -x[0])
        best_score, best_input, best_output, best_outcomes = scored[0]

        if not should_ask(candidates, weights, run_candidate, func_name, best_input, best_output, gamma):
            break

        answer = oracle_answer(reference_code, func_name, best_input, best_output, run_candidate)
        questions_asked.append({
            "input": best_input, "hypothesized_output": best_output,
            "answer": answer, "score": best_score
        })

        weights = update_weights(weights, best_outcomes, answer, epsilon)
        test_pool.remove((best_input, best_output))

    best_idx = max(weights, key=weights.get)
    return {
        "task_id": sample.get("task_id"),
        "n_questions_asked": len(questions_asked),
        "questions": questions_asked,
        "final_candidate": candidates[best_idx],
        "final_weights": weights
    }

In [18]:
with open("../results/phase2_ambiguity_7b.json", "r") as f:
    results_phase2_7b = json.load(f)

In [ ]:
ambiguous_task_ids = [r["task_id"] for r in results_phase2_7b if r.get("n_disagreement_points", 0) > 0]
print(f"Running Phase 3 on {len(ambiguous_task_ids)} ambiguous tasks: {ambiguous_task_ids}")

results_phase3 = []

for r in results_phase2_7b:
    if r.get("n_disagreement_points", 0) <= 0:
        continue

    task_id = r["task_id"]
    sample = next(s for s in dataset["train"] if s["task_id"] == task_id)
    candidates = r["candidates"]

    print(f"Running clarification loop on task {task_id}...")
    clarif_result = clarification_loop(sample, candidates, k_max=4)

    passed, error = evaluate(
        f"```python\n{clarif_result['final_candidate']}\n```",
        sample["test_list"],
        setup_code=sample.get("test_setup_code", "")
    )

    clarif_result["passed"] = passed
    clarif_result["error"] = error
    results_phase3.append(clarif_result)

    with open("../results/phase3_clarification.json", "w") as f:
        json.dump(results_phase3, f, indent=2)

print("Done.")
n_passed = sum(1 for r in results_phase3 if r["passed"])
print(f"\nPost-clarification pass rate: {n_passed}/{len(results_phase3)} ({n_passed/len(results_phase3)*100:.1f}%)")

Running Phase 3 on 15 ambiguous tasks: [602, 603, 607, 608, 609, 610, 612, 617, 619, 622, 623, 625, 626, 627, 630]
Running clarification loop on task 602...
Running clarification loop on task 603...
Running clarification loop on task 607...
Running clarification loop on task 608...
Running clarification loop on task 609...
Running clarification loop on task 610...
Running clarification loop on task 612...
Running clarification loop on task 617...
Running clarification loop on task 619...
Running clarification loop on task 622...
Running clarification loop on task 623...
Running clarification loop on task 625...
Running clarification loop on task 626...
Running clarification loop on task 627...
Running clarification loop on task 630...
Done.

Post-clarification pass rate: 8/15 (53.3%)


In [20]:
baseline_on_subset = [r for r in results_7b_baseline if r["task_id"] in ambiguous_task_ids]
baseline_passed = sum(1 for r in baseline_on_subset if r["passed"])
print(f"Baseline (one-shot) pass rate on same subset: {baseline_passed}/{len(baseline_on_subset)} ({baseline_passed/len(baseline_on_subset)*100:.1f}%)")

Baseline (one-shot) pass rate on same subset: 5/15 (33.3%)


In [21]:
avg_q = sum(r["n_questions_asked"] for r in results_phase3) / len(results_phase3)
print(f"Average questions asked: {avg_q:.2f}")

Average questions asked: 2.20


## Testing on 50 sequential tasks in MBPP

#### Baseline on 50 sequential tasks

In [22]:
from tqdm import tqdm

results_7b_baseline_50 = []
passed = 0

for i in tqdm(range(30, 80)):
    sample = dataset["train"][i]
    try:
        response = generate(sample["text"], test_example=sample["test_list"][0])
    except Exception as e:
        results_7b_baseline_50.append({"idx": i, "task_id": sample.get("task_id", i), "passed": False, "error": f"generation error: {e}"})
        continue

    success, error = evaluate(response, sample["test_list"], setup_code=sample.get("test_setup_code", ""))
    if success:
        passed += 1

    results_7b_baseline_50.append({
        "idx": i, "task_id": sample.get("task_id", i), "passed": success,
        "error": error, "code": extract_code(response)
    })

    with open("../results/baseline_7b_run_50_new.json", "w") as f:
        json.dump(results_7b_baseline_50, f, indent=2)

print(f"\nBaseline pass rate: {passed}/50 ({passed/50*100:.1f}%)")




















































100%|██████████| 50/50 [02:30<00:00,  3.01s/it]


Baseline pass rate: 33/50 (66.0%)


#### Ambiguity analysis on the same 50 tasks

In [23]:
results_phase2_50 = []

for i in range(30, 80):
    sample = dataset["train"][i]
    print(f"Analyzing task {sample.get('task_id', i)}...")
    result = analyze_ambiguity(sample, n_candidates=8, temperature=0.8)
    results_phase2_50.append(result)

    with open("../results/phase2_ambiguity_50_new.json", "w") as f:
        json.dump(results_phase2_50, f, indent=2)

print("Done.")

Analyzing task 631...
Analyzing task 632...
Analyzing task 633...
Analyzing task 634...
Analyzing task 635...
Analyzing task 636...
Analyzing task 637...
Analyzing task 638...
Analyzing task 639...
Analyzing task 640...
Analyzing task 641...
Analyzing task 642...
Analyzing task 643...
Analyzing task 644...
Analyzing task 645...
Analyzing task 646...
Analyzing task 647...
Analyzing task 648...
Analyzing task 649...
Analyzing task 650...
Analyzing task 651...
Analyzing task 652...
Analyzing task 653...
Analyzing task 654...
Analyzing task 655...
Analyzing task 656...
Analyzing task 657...
Analyzing task 658...
Analyzing task 659...
Analyzing task 660...
Analyzing task 661...
Analyzing task 662...
Analyzing task 663...
Analyzing task 664...
Analyzing task 665...
Analyzing task 666...
Analyzing task 667...
Analyzing task 668...
Analyzing task 669...
Analyzing task 670...
Analyzing task 671...
Analyzing task 672...
Analyzing task 673...
Analyzing task 674...
Analyzing task 675...
Analyzing 

In [24]:
ambiguous_task_ids_50 = [r["task_id"] for r in results_phase2_50 if r.get("n_disagreement_points", 0) > 0]
print(f"Running Phase 3 on {len(ambiguous_task_ids_50)} ambiguous tasks")

results_phase3_50 = []

for r in results_phase2_50:
    if r.get("n_disagreement_points", 0) <= 0:
        continue

    task_id = r["task_id"]
    sample = next(s for s in dataset["train"] if s["task_id"] == task_id)
    candidates = r["candidates"]

    clarif_result = clarification_loop(sample, candidates, k_max=4)
    passed, error = evaluate(
        f"```python\n{clarif_result['final_candidate']}\n```",
        sample["test_list"],
        setup_code=sample.get("test_setup_code", "")
    )
    clarif_result["passed"] = passed
    clarif_result["error"] = error
    results_phase3_50.append(clarif_result)

    with open("../results/phase3_clarification_50.json", "w") as f:
        json.dump(results_phase3_50, f, indent=2)

print("Done.")

Running Phase 3 on 25 ambiguous tasks
Done.


In [ ]:
all_phase3 = results_phase3 + results_phase3_50  
all_baseline = results_7b_baseline + results_7b_baseline_50

baseline_dict = {r["task_id"]: r["passed"] for r in all_baseline}
clarif_task_ids = [r["task_id"] for r in all_phase3]

baseline_on_ambiguous = sum(1 for tid in clarif_task_ids if baseline_dict.get(tid))
clarif_passed = sum(1 for r in all_phase3 if r["passed"])

print(f"Total ambiguous tasks tested: {len(all_phase3)}")
print(f"Baseline pass rate: {baseline_on_ambiguous}/{len(all_phase3)} ({baseline_on_ambiguous/len(all_phase3)*100:.1f}%)")
print(f"Clarification pass rate: {clarif_passed}/{len(all_phase3)} ({clarif_passed/len(all_phase3)*100:.1f}%)")

Total ambiguous tasks tested: 40
Baseline pass rate: 15/40 (37.5%)
Clarification pass rate: 28/40 (70.0%)


## Testing on 50 tasks sampled at random

In [ ]:
import random
random.seed(69)

all_indices = list(range(len(dataset["train"])))
already_used = set(range(30))  
available = [i for i in all_indices if i not in already_used]

sample_indices_rand = random.sample(available, 50)
print(sample_indices_rand)

[49, 79, 115, 64, 339, 206, 197, 311, 242, 267, 249, 320, 254, 295, 194, 316, 310, 255, 102, 241, 230, 62, 176, 105, 130, 188, 138, 368, 287, 136, 82, 80, 228, 348, 63, 164, 340, 220, 50, 78, 141, 45, 104, 157, 155, 170, 107, 275, 350, 71]


#### Baseline on 50 random tasks

In [28]:
results_7b_baseline_rand = []
passed = 0

for i in tqdm(sample_indices_rand):
    sample = dataset["train"][i]
    try:
        response = generate(sample["text"], test_example=sample["test_list"][0])
    except Exception as e:
        results_7b_baseline_rand.append({"idx": i, "task_id": sample.get("task_id", i), "passed": False, "error": f"generation error: {e}"})
        continue

    success, error = evaluate(response, sample["test_list"], setup_code=sample.get("test_setup_code", ""))
    if success:
        passed += 1

    results_7b_baseline_rand.append({
        "idx": i, "task_id": sample.get("task_id", i), "passed": success,
        "error": error, "code": extract_code(response)
    })

    with open("../results/baseline_7b_run_50_random.json", "w") as f:
        json.dump(results_7b_baseline_rand, f, indent=2)

print(f"\nBaseline pass rate: {passed}/50 ({passed/50*100:.1f}%)")




















































100%|██████████| 50/50 [03:03<00:00,  3.67s/it]


Baseline pass rate: 42/50 (84.0%)


#### Ambiguity analysis on the 50 random tasks

In [29]:
results_phase2_rand = []

for i in sample_indices_rand:
    sample = dataset["train"][i]
    print(f"Analyzing task {sample.get('task_id', i)}...")
    result = analyze_ambiguity(sample, n_candidates=8, temperature=0.8)
    results_phase2_rand.append(result)

    with open("../results/phase2_ambiguity_50_random.json", "w") as f:
        json.dump(results_phase2_rand, f, indent=2)

print("Done.")

Analyzing task 650...
Analyzing task 680...
Analyzing task 716...
Analyzing task 665...
Analyzing task 940...
Analyzing task 807...
Analyzing task 798...
Analyzing task 912...
Analyzing task 843...
Analyzing task 868...
Analyzing task 850...
Analyzing task 921...
Analyzing task 855...
Analyzing task 896...
Analyzing task 795...
Analyzing task 917...
Analyzing task 911...
Analyzing task 856...
Analyzing task 703...
Analyzing task 842...
Analyzing task 831...
Analyzing task 663...
Analyzing task 777...
Analyzing task 706...
Analyzing task 731...
Analyzing task 789...
Analyzing task 739...
Analyzing task 969...
Analyzing task 888...
Analyzing task 737...
Analyzing task 683...
Analyzing task 681...
Analyzing task 829...
Analyzing task 949...
Analyzing task 664...
Analyzing task 765...
Analyzing task 941...
Analyzing task 821...
Analyzing task 651...
Analyzing task 679...
Analyzing task 742...
Analyzing task 646...
Analyzing task 705...
Analyzing task 758...
Analyzing task 756...
Analyzing 

#### Clarification loop on these 50 random tasks

In [30]:
ambiguous_task_ids_rand = [r["task_id"] for r in results_phase2_rand if r.get("n_disagreement_points", 0) > 0]
print(f"Running Phase 3 on {len(ambiguous_task_ids_rand)} ambiguous tasks")

results_phase3_rand = []

for r in results_phase2_rand:
    if r.get("n_disagreement_points", 0) <= 0:
        continue

    task_id = r["task_id"]
    sample = next(s for s in dataset["train"] if s["task_id"] == task_id)
    candidates = r["candidates"]

    clarif_result = clarification_loop(sample, candidates, k_max=4)
    passed, error = evaluate(
        f"```python\n{clarif_result['final_candidate']}\n```",
        sample["test_list"],
        setup_code=sample.get("test_setup_code", "")
    )
    clarif_result["passed"] = passed
    clarif_result["error"] = error
    results_phase3_rand.append(clarif_result)

    with open("../results/phase3_clarification_50_random.json", "w") as f:
        json.dump(results_phase3_rand, f, indent=2)

print("Done.")

Running Phase 3 on 21 ambiguous tasks
Done.


In [31]:
all_phase3 = results_phase3 + results_phase3_50 + results_phase3_rand
all_baseline = results_7b_baseline + results_7b_baseline_50 + results_7b_baseline_rand

baseline_dict = {r["task_id"]: r["passed"] for r in all_baseline}
clarif_task_ids = [r["task_id"] for r in all_phase3]

baseline_on_ambiguous = sum(1 for tid in clarif_task_ids if baseline_dict.get(tid))
clarif_passed = sum(1 for r in all_phase3 if r["passed"])

print(f"Total ambiguous tasks tested: {len(all_phase3)}")
print(f"Baseline pass rate: {baseline_on_ambiguous}/{len(all_phase3)} ({baseline_on_ambiguous/len(all_phase3)*100:.1f}%)")
print(f"Clarification pass rate: {clarif_passed}/{len(all_phase3)} ({clarif_passed/len(all_phase3)*100:.1f}%)")

Total ambiguous tasks tested: 61
Baseline pass rate: 31/61 (50.8%)
Clarification pass rate: 48/61 (78.7%)


In [ ]:
def load_json(path):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Warning: {path} not found, skipping")
        return []

baseline_15b_100 = load_json("../results/baseline_run_20.json") 
baseline_7b_30 = load_json("../results/baseline_7b_run_30.json")
baseline_7b_50seq = load_json("../results/baseline_7b_run_50_new.json")
baseline_7b_50rand = load_json("../results/baseline_7b_run_50_random.json")

phase2_15b_30 = load_json("../results/phase2_ambiguity_analysis.json")
phase2_7b_30 = load_json("../results/phase2_ambiguity_7b.json")
phase2_7b_50seq = load_json("../results/phase2_ambiguity_50_new.json")
phase2_7b_50rand = load_json("../results/phase2_ambiguity_50_random.json")

phase3_15 = load_json("../results/phase3_clarification.json")
phase3_50seq = load_json("../results/phase3_clarification_50.json")
phase3_50rand = load_json("../results/phase3_clarification_50_random.json")

def pass_rate(results):
    if not results:
        return 0, 0, 0.0
    n = len(results)
    p = sum(1 for r in results if r.get("passed"))
    return p, n, p / n * 100


def ambiguity_rate(phase2_results):
    analyzable = [r for r in phase2_results if "n_disagreement_points" in r]
    if not analyzable:
        return 0, 0, 0.0
    n = len(analyzable)
    amb = sum(1 for r in analyzable if r["n_disagreement_points"] > 0)
    return amb, n, amb / n * 100


print("=" * 60)
print("PHASE 1 — BASELINE (one-shot pass@1)")
print("=" * 60)
p, n, r = pass_rate(baseline_15b_100)
print(f"1.5B, N={n}: {p}/{n} ({r:.1f}%)")

p, n, r = pass_rate(baseline_7b_30)
print(f"7B, sequential N={n}: {p}/{n} ({r:.1f}%)")

p, n, r = pass_rate(baseline_7b_50seq)
print(f"7B, sequential (2nd batch) N={n}: {p}/{n} ({r:.1f}%)")

p, n, r = pass_rate(baseline_7b_50rand)
print(f"7B, random (seed=69) N={n}: {p}/{n} ({r:.1f}%)")

all_7b_baseline = baseline_7b_30 + baseline_7b_50seq + baseline_7b_50rand
p, n, r = pass_rate(all_7b_baseline)
print(f"7B, ALL COMBINED N={n}: {p}/{n} ({r:.1f}%)")


print("\n" + "=" * 60)
print("PHASE 2 — AMBIGUITY DETECTION RATE")
print("=" * 60)
a, n, r = ambiguity_rate(phase2_15b_30)
print(f"1.5B, N={n}: {a}/{n} tasks show disagreement ({r:.1f}%)")

a, n, r = ambiguity_rate(phase2_7b_30)
print(f"7B, sequential N={n}: {a}/{n} tasks show disagreement ({r:.1f}%)")

a, n, r = ambiguity_rate(phase2_7b_50seq)
print(f"7B, sequential (2nd) N={n}: {a}/{n} tasks show disagreement ({r:.1f}%)")

a, n, r = ambiguity_rate(phase2_7b_50rand)
print(f"7B, random N={n}: {a}/{n} tasks show disagreement ({r:.1f}%)")


print("\n" + "=" * 60)
print("PHASE 3 — CLARIFICATION vs BASELINE (on ambiguous subset only)")
print("=" * 60)

all_phase3 = phase3_15 + phase3_50seq + phase3_50rand
all_baseline = baseline_7b_30 + baseline_7b_50seq + baseline_7b_50rand
baseline_dict = {r["task_id"]: r["passed"] for r in all_baseline}

clarif_task_ids = [r["task_id"] for r in all_phase3]
baseline_on_ambiguous = sum(1 for tid in clarif_task_ids if baseline_dict.get(tid))
clarif_passed = sum(1 for r in all_phase3 if r["passed"])
n_total = len(all_phase3)

print(f"Total ambiguous tasks tested: {n_total}")
print(f"Baseline (one-shot) pass rate:      {baseline_on_ambiguous}/{n_total} ({baseline_on_ambiguous/n_total*100:.1f}%)")
print(f"Clarification-based pass rate:      {clarif_passed}/{n_total} ({clarif_passed/n_total*100:.1f}%)")
print(f"Absolute improvement:               +{(clarif_passed - baseline_on_ambiguous)/n_total*100:.1f} points")

avg_q = sum(r["n_questions_asked"] for r in all_phase3) / n_total
print(f"Average clarification questions asked: {avg_q:.2f}")

# breakdown by batch, in case sequential vs random differ meaningfully
for name, p3, bl in [("Original 15", phase3_15, baseline_7b_30),
                      ("Sequential 50", phase3_50seq, baseline_7b_50seq),
                      ("Random 50", phase3_50rand, baseline_7b_50rand)]:
    bl_dict = {r["task_id"]: r["passed"] for r in bl}
    tids = [r["task_id"] for r in p3]
    if not tids:
        continue
    bl_pass = sum(1 for t in tids if bl_dict.get(t))
    cl_pass = sum(1 for r in p3 if r["passed"])
    print(f"  {name}: baseline {bl_pass}/{len(tids)} ({bl_pass/len(tids)*100:.1f}%) -> clarification {cl_pass}/{len(tids)} ({cl_pass/len(tids)*100:.1f}%)")

PHASE 1 — BASELINE (one-shot pass@1)
1.5B, N=100: 51/100 (51.0%)
7B, sequential N=30: 20/30 (66.7%)
7B, sequential (2nd batch) N=50: 33/50 (66.0%)
7B, random (seed=69) N=50: 42/50 (84.0%)
7B, ALL COMBINED N=130: 95/130 (73.1%)

PHASE 2 — AMBIGUITY DETECTION RATE
1.5B, N=29: 19/29 tasks show disagreement (65.5%)
7B, sequential N=29: 15/29 tasks show disagreement (51.7%)
7B, sequential (2nd) N=50: 25/50 tasks show disagreement (50.0%)
7B, random N=49: 21/49 tasks show disagreement (42.9%)

PHASE 3 — CLARIFICATION vs BASELINE (on ambiguous subset only)
Total ambiguous tasks tested: 61
Baseline (one-shot) pass rate:      31/61 (50.8%)
Clarification-based pass rate:      48/61 (78.7%)
Absolute improvement:               +27.9 points
Average clarification questions asked: 1.77
  Original 15: baseline 5/15 (33.3%) -> clarification 8/15 (53.3%)
  Sequential 50: baseline 10/25 (40.0%) -> clarification 20/25 (80.0%)
  Random 50: baseline 16/21 (76.2%) -> clarification 20/21 (95.2%)


In [35]:
import math
def wilson_ci(p, n, z=1.96):
    se = math.sqrt(p * (1 - p) / n)
    return max(0, p - z*se), min(1, p + z*se)

baseline_p, clarif_p, n = 31/61, 48/61, 61
print(f"Baseline 95% CI: {wilson_ci(baseline_p, n)}")
print(f"Clarification 95% CI: {wilson_ci(clarif_p, n)}")

Baseline 95% CI: (0.38273744065067694, 0.6336560019722739)
Clarification 95% CI: (0.6841182612820893, 0.8896522305211894)


## Phase 3 Summary — EIG-Based Clarification vs. One-Shot Baseline

### Headline Result

On a combined pool of **61 tasks flagged as ambiguous** by Phase 2's candidate-disagreement analysis
(drawn from an original 15-task pilot, a 50-task sequential batch, and a 50-task random batch, seed=69),
EIG-based clarification questioning improves pass@1 substantially over one-shot generation:

| Strategy | Pass@1 | 95% CI |
|---|---|---|
| One-shot baseline | 31/61 (50.8%) | (38.3%, 63.4%) |
| EIG-based clarification | 48/61 (78.7%) | (68.4%, 89.0%) |

**Absolute improvement: +27.9 points.** The two confidence intervals do not overlap, indicating the
improvement is unlikely to be attributable to sampling noise. Clarification asked an average of
**1.77 questions per task** (out of a max of 4), consistent with the reference paper's finding that
most of the benefit comes from a small number of well-chosen questions rather than exhaustive querying.

This result is directionally consistent with Garg, Kim & Xi (2026), who report a similar-magnitude
improvement on MBPP+/HumanEval+ ambiguous-prompt subsets using the same EIG-based approach on
Qwen 2.5 7B and Llama 3.1 8B.

### Pipeline Summary

1. **Phase 1 (Baseline):** One-shot generation, greedy decoding, evaluated via a custom harness
   (function-name/signature aliasing, AST-based code extraction, exec-based test execution).
2. **Phase 2 (Ambiguity detection):** For each problem, sample 8 candidates at `temperature=0.8`,
   filter invalid/duplicate candidates via behavioral deduplication, and flag genuine ambiguity where
   surviving candidates disagree on real probe inputs drawn from MBPP's hidden test cases.
3. **Phase 3 (Clarification):** For flagged-ambiguous problems, build a candidate test pool directly
   from Phase 2's disagreement points, score each by expected information gain (Section 4.2–4.3 of the
   reference paper), and iteratively ask the highest-scoring question (answered by the MBPP reference
   solution, acting as oracle) up to `k_max=4` times, using an ask-or-submit stopping rule
   (Section 4.4). Final answer = highest-posterior-weight candidate after all questions.

### Model Comparison (Phase 1)

| Model | N | Pass@1 |
|---|---|---|
| Qwen2.5-Coder-1.5B-Instruct | 100 | 51.0% |
| Qwen2.5-Coder-7B-Instruct (4-bit) | 130 (combined) | 73.1% |

Switching from 1.5B to 7B was motivated by an observed noise problem in Phase 2: on 1.5B, a meaningful
share of "disagreement" between candidates stemmed from runtime crashes and basic competence errors
(missing imports, wrong algorithms) rather than genuine ambiguity. On 7B, no crash-driven disagreement
was observed in the same analysis — candidates that disagreed did so by producing different **valid**
outputs, indicating a cleaner ambiguity signal for the EIG pipeline to work with.

### Known Limitations / Scope Decisions

- **Clarification test pool is derived from Phase 2's disagreement points**, not generated by the LLM
  as a separate step (unlike the paper's full pipeline). This is a deliberate scoping simplification —
  test *scoring* (the intellectual core of the EIG method) is implemented faithfully; test *generation*
  reuses signal already computed in Phase 2 rather than adding a new LLM-generation stage.
- **Custom-class equality comparison is incomplete.** Candidates returning instances of model-defined
  classes without `__repr__` (e.g. a `Pair` class) can produce spurious disagreement due to
  default memory-address reprs. A partial fix (normalizing generator/map outputs) was applied after
  the initial 1.5B Phase 2 run and is present in all 7B runs; the custom-class case remains unresolved
  and is a candidate for future work.
- **Dataset ordering effects.** The initial 30-task validation batch and one 50-task batch used
  sequential MBPP indices (601–630, 631–680) for development speed; both showed a notably lower
  baseline pass rate (~66%) than the later random 50-task sample (84%, seed=69), suggesting MBPP's
  task ordering is not difficulty-neutral. The combined 130-task baseline (73.1%) is reported as the
  primary Phase 1 number to average over this effect.
- **Oracle is the MBPP reference solution**, not a live human, matching the reference paper's own
  evaluation setup (not a live human oracle either).